# ChakraModel v8 Verification (CPU-only, no GPU quota used)

v7's bug (found and confirmed via `__file__` inspection, then reproduced in an offline unit test): the frame cap was applied PER INDIVIDUAL VIDEO FILE, not per dataset folder. Hyperkvasir's folders each hold ~139 short clips, so capping every clip at 2000 still summed to ~278,000 total. v8 fixes this: the budget (2000) is now a TOTAL per dataset folder, split proportionally across however many video files it contains.

1. **Run \u2192 Factory reset session** first.
2. Attach the v8 code dataset, weights dataset, polyp datasets.
3. **Accelerator: None / CPU.**
4. Run All. Paste the full output back — expect each video dataset's frame_count to land near 2000 (not 277,921/280,312), and total across everything near ~5,552.

In [ ]:
import os, shutil, glob

WORK = '/kaggle/working'

code_zip = None
for p in glob.glob('/kaggle/input/**/ChakraModel_EvalHarness_Code*.zip', recursive=True):
    code_zip = p
    break

if code_zip:
    print('Unpacking code from', code_zip)
    shutil.unpack_archive(code_zip, WORK)
else:
    eh_dir = None
    for root, dirs, files in os.walk('/kaggle/input'):
        if os.path.basename(root) == 'eval_harness' and 'run_eval.py' in files:
            eh_dir = root
            break
    assert eh_dir, (
        'Could not find ChakraModel_EvalHarness_Code*.zip or an extracted eval_harness/ '
        'folder anywhere under /kaggle/input — attach the code dataset.'
    )
    dataset_root = os.path.dirname(eh_dir)
    print('Found already-extracted code at', dataset_root)
    shutil.copytree(eh_dir, f'{WORK}/eval_harness', dirs_exist_ok=True)
    src_dir = os.path.join(dataset_root, 'src')
    if os.path.isdir(src_dir):
        shutil.copytree(src_dir, f'{WORK}/src', dirs_exist_ok=True)
    print('Copied eval_harness/ and src/ into', WORK)

os.makedirs(f'{WORK}/weights/checkpoints', exist_ok=True)
wanted = {'chakra_transformer_best.pth', 'combo1_best.pth'}
found = set()
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        if f in wanted and f not in found:
            src = os.path.join(root, f)
            dst = f'{WORK}/weights/checkpoints/{f}'
            shutil.copy(src, dst)
            found.add(f)
            print('Linked', f, 'from', src)
missing = wanted - found
if missing:
    print('WARNING missing weights:', missing)
else:
    print('Both checkpoints staged OK.')

In [ ]:
!pip install -q opencv-python-headless

In [ ]:
# ============================================================
# CPU-ONLY VERIFICATION — no model loading, no GPU needed.
# ============================================================
import sys, os

WORK = '/kaggle/working'
sys.path.insert(0, f'{WORK}/eval_harness')
sys.path.insert(0, f'{WORK}/src')

print("="*70)
print("1. FILE VERSION CHECK")
print("="*70)

def check(path, must_contain, label):
    if not os.path.exists(path):
        print(f"  [MISSING] {path}")
        return
    src = open(path).read()
    ok = all(s in src for s in must_contain)
    print(f"  [{'OK' if ok else 'FAIL'}] {label}: {os.path.realpath(path)}")
    for s in must_contain:
        print(f"       {'found' if s in src else 'MISSING'}: {s!r}")

check(f'{WORK}/eval_harness/resource_monitor.py',
      ['MAX_BATCH         = 8'], 'batch ceiling fix (v4+)')
check(f'{WORK}/eval_harness/eval_engine.py',
      ['mode="default"', '_key_renames', 'ppd_pred.weight'], 'compile-mode + checkpoint-remap fix (v4/v6+)')
check(f'{WORK}/eval_harness/dataset_discovery.py',
      ['_discover_recursive', 'MAX_FRAMES_PER_DATASET', '_sampling_plan', 'IterableDataset'],
      'recursive discovery + PER-DATASET video sampling budget fix (v8)')

print()
print("="*70)
print("2. WHICH FILE DOES `import dataset_discovery` ACTUALLY LOAD?")
print("="*70)
for _mod in list(sys.modules):
    if _mod == 'dataset_discovery':
        print(f"  [NOTE] Evicting cached module '{_mod}' from sys.modules before import.")
        del sys.modules[_mod]
import dataset_discovery
print(f"  LOADED FROM: {dataset_discovery.__file__}")
print(f"  MAX_FRAMES_PER_DATASET value: {getattr(dataset_discovery, 'MAX_FRAMES_PER_DATASET', 'MISSING')}")

print()
print("="*70)
print("3. ACTUAL discover() BEHAVIOR (real scan, no GPU)")
print("="*70)
metas = dataset_discovery.discover('/kaggle/input')
total = 0
for m in metas:
    print(f"  {m.name:45s} | {m.kind.name:15s} | {m.frame_count:8d} frames")
    total += m.frame_count
print(f"\n  TOTAL FRAMES ACROSS ALL DATASETS: {total:,}")
if total > 20000:
    print("\n  [WARNING] Still not bounded as expected — paste this full output back before running GPU eval.")
else:
    print("\n  Looks correctly bounded — safe to proceed to the real GPU run.")

print()
print("="*70)
print("4. WEIGHTS PRESENT?")
print("="*70)
for f in ['chakra_transformer_best.pth', 'combo1_best.pth']:
    p = f'{WORK}/weights/checkpoints/{f}'
    print(f"  {'OK' if os.path.exists(p) else 'MISSING'}: {p}")